In [ ]:
from __future__ import annotations

from libraries import *
from parameters import *
from util import *


from typing import Iterable, Optional, Dict
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats


In [ ]:
adata = sc.read_h5ad("/home/eraslab1/Projects/AbbasScreen/Data/ComboScreen.h5ad")


In [ ]:
adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

In [ ]:
cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)


In [ ]:
sc.pp.normalize_total(adata, target_sum=20000)
sc.pp.log1p(adata)


In [ ]:
humanTFs = pd.read_csv("./../Human_TFs.csv")

In [ ]:
humanTFs=humanTFs.loc[humanTFs.TFName.isin(adata.var_names),]

In [ ]:
myTFs = list(humanTFs.TFName)

In [ ]:
from __future__ import annotations

from typing import Iterable, Optional
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats


def _get_X(adata, layer: Optional[str] = None):
    # Your data are already normalized + log-transformed,
    # so by default we use adata.X (or a specified layer if you pass one).
    return adata.X if layer is None else adata.layers[layer]


def _col_as_dense(X, j: int) -> np.ndarray:
    v = X[:, j]
    if sp.issparse(v):
        return np.asarray(v.toarray()).ravel()
    return np.asarray(v).ravel()


def _bh_fdr(p: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg FDR for 1D p-value array."""
    p = np.asarray(p, dtype=float)
    m = p.size
    order = np.argsort(p)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, m + 1)
    q = p * m / ranks
    q_sorted = q[order]
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    out = np.empty_like(q_sorted)
    out[order] = np.minimum(q_sorted, 1.0)
    return out


def differential_tf_expression_by_final_label(
    adata,
    tfs: Iterable[str],
    group_key: str = "final_label",
    layer: Optional[str] = None,     # None -> adata.X (already normalized+log)
    min_cells_in_group: int = 25,
    min_cells_outside: int = 50,
    test: str = "wilcoxon",          # "wilcoxon" (Mann–Whitney) or "ttest"
    direction: str = "up",           # "up" (group higher) or "two-sided"
    effect: str = "mean_diff",       # "mean_diff" (recommended for log data) or "log2fc"
    eps: float = 1e-9,
) -> pd.DataFrame:
    """
    For each TF and each final_label, test whether TF expression is higher in that label
    vs all other cells, using ALREADY normalized + log-transformed values.

    Returns a long DataFrame with:
      TF, group, n_in, n_out, mean_in, mean_out, frac_expr_in, frac_expr_out,
      effect_size, pval, fdr

    Effect size:
      - effect="mean_diff": mean_in - mean_out   (best for log1p-ish data)
      - effect="log2fc":    log2((mean_in+eps)/(mean_out+eps))  (less interpretable on log data)
    """

    if group_key not in adata.obs:
        raise KeyError(f"{group_key} not found in adata.obs")

    X = _get_X(adata, layer=layer)
    var_names = adata.var_names

    # Map TFs to indices (drop missing)
    tfs = list(dict.fromkeys([str(x) for x in tfs]))  # unique preserve order
    present = [tf for tf in tfs if tf in var_names]
    missing = [tf for tf in tfs if tf not in var_names]
    if len(present) == 0:
        raise ValueError("None of the TFs were found in adata.var_names.")

    tf_idx = {tf: int(var_names.get_loc(tf)) for tf in present}

    groups = adata.obs[group_key].astype(str)
    uniq_groups = groups.unique().tolist()

    results = []

    for g in uniq_groups:
        in_mask = (groups.values == g)
        out_mask = ~in_mask

        n_in = int(in_mask.sum())
        n_out = int(out_mask.sum())

        if n_in < min_cells_in_group or n_out < min_cells_outside:
            continue

        for tf in present:
            j = tf_idx[tf]
            x = _col_as_dense(X, j)

            xin = x[in_mask]
            xout = x[out_mask]

            mean_in = float(np.mean(xin))
            mean_out = float(np.mean(xout))
            frac_in = float(np.mean(xin > 0))
            frac_out = float(np.mean(xout > 0))

            # Effect size for log-normalized data
            if effect == "mean_diff":
                eff = float(mean_in - mean_out)

            # Statistical test
            if test.lower() in ("wilcoxon", "mannwhitney", "mw"):
                alt = "greater" if direction == "up" else "two-sided"
                pval = float(stats.mannwhitneyu(xin, xout, alternative=alt, method="asymptotic").pvalue)
            elif test.lower() in ("ttest", "t", "t-test"):
                pval = float(stats.ttest_ind(xin, xout, equal_var=False, nan_policy="omit").pvalue)
            else:
                raise ValueError("test must be 'wilcoxon' (Mann–Whitney) or 'ttest'")

            results.append({
                "TF": tf,
                "group": g,
                "n_in": n_in,
                "n_out": n_out,
                "mean_in": mean_in,
                "mean_out": mean_out,
                "frac_expr_in": frac_in,
                "frac_expr_out": frac_out,
                "effect_size": eff,
                "effect_type": effect,
                "pval": pval,
            })

    res = pd.DataFrame(results)
    if res.empty:
        raise ValueError(
            "No tests were run (likely due to min_cells_in_group/min_cells_outside thresholds). "
            "Lower thresholds or check group sizes."
        )

    # FDR within each group (common for "markers per cluster")
    res["fdr"] = np.nan
    for g in res["group"].unique():
        m = res["group"] == g
        res.loc[m, "fdr"] = _bh_fdr(res.loc[m, "pval"].to_numpy())

    # Rank by significance then effect
    res = res.sort_values(["group", "fdr", "effect_size"], ascending=[True, True, False]).reset_index(drop=True)

    if missing:
        res.attrs["missing_tfs"] = missing

    return res


# -------------------------
# Example usage
# -------------------------

res = differential_tf_expression_by_final_label(
    adata,
    tfs=myTFs,
    group_key="final_label",
    layer=None,               # uses adata.X which you said is already normalized+log
    min_cells_in_group=100,
    test="wilcoxon",
    direction="up",
    effect="mean_diff",       # <-- updated for log-normalized input
)

# res.head()

In [ ]:
res_sgn = res.loc[res.fdr < 0.01,:]

In [ ]:
res_sgn.to_csv("tf_differential_expression_by_final_label.tsv", sep="\t", index=False)

In [ ]:
res_sgn

In [ ]:
# res_sgn is assumed to have columns: ["TF", "group", "log2fc_mean", ...]
# and adata.obs["final_label"] exists

group_key = "final_label"

# 1) Top 20 TFs per group by log2fc_mean
top_per_group = (
    res_sgn.sort_values(["group", "log2fc_mean"], ascending=[True, False])
           .groupby("group", sort=False)
           .head(20)
           .copy()
)

# 2) Build TF order (group blocks: 20 TFs for group1, then 20 TFs for group2, ...)
#    and keep unique while preserving order
tf_order = []
seen = set()
for g, sub in top_per_group.groupby("group", sort=False):
    for tf in sub["TF"].tolist():
        if tf not in seen:
            tf_order.append(tf)
            seen.add(tf)

# Optional: sanity check TFs exist in adata.var_names (or adata.raw.var_names)
present = [tf for tf in tf_order if tf in adata.var_names]
missing = [tf for tf in tf_order if tf not in adata.var_names]
if missing:
    print(f"Warning: {len(missing)} TFs not found in adata.var_names. Dropping them. Examples: {missing[:10]}")
tf_order = present

# 3) Ensure group ordering on x-axis (optional: order by mean score or just alphabetical)
# Here: use observed order of categories if categorical, else sort.
if pd.api.types.is_categorical_dtype(adata.obs[group_key]):
    group_order = list(adata.obs[group_key].cat.categories)
else:
    group_order = sorted(adata.obs[group_key].astype(str).unique().tolist())

adata.obs[group_key] = adata.obs[group_key].astype("category")
adata.obs[group_key] = adata.obs[group_key].cat.set_categories(group_order, ordered=True)

# 4) Plot dotplot
# - if you want to plot a layer (e.g., "log1p"), pass layer="log1p"
# - standard_scale="var" makes rows comparable (optional)
sc.pl.dotplot(
    adata,
    var_names=tf_order,
    groupby=group_key,
    standard_scale="var",   # optional; remove if you want raw scale
    swap_axes=True,         # makes x=groups, y=genes (TFs)
    dendrogram=False,
)

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, m + 1)
    q = pvals * m / ranks
    q_sorted = q[order]
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    out = np.empty_like(q_sorted)
    out[order] = np.minimum(q_sorted, 1.0)
    return out


def clean_perturbation_labels(pert_series):
    """
    Removes 'NTC' from combinatorial perturbations.
    Examples:
        'A+NTC'     -> 'A'
        'NTC+A'     -> 'A'
        'A+NTC+B'   -> 'A+B'
        'NTC'       -> 'NTC'
    """

    pert = pert_series.astype(str).copy()

    def remove_ntc(label):
        parts = label.split("+")
        parts_clean = [p for p in parts if p != "NTC"]
        if len(parts_clean) == 0:
            return "NTC"
        return "+".join(parts_clean)

    pert = pert.apply(remove_ntc)

    return pert


def get_tf_matrix(adata, tfs, layer=None):
    var_names = adata.var_names
    present = [tf for tf in tfs if tf in var_names]
    if not present:
        raise ValueError("None of the TFs found in var_names.")

    idx = [var_names.get_loc(tf) for tf in present]
    X = adata.X if layer is None else adata.layers[layer]

    if sp.issparse(X):
        mat = X[:, idx].toarray()
    else:
        mat = np.asarray(X[:, idx])

    return present, mat


def linear_model_tf_vs_ntc(
    adata,
    tfs,
    perturb_key="perturbation",
    control="NTC",
    layer=None,
    alpha=0.01,
):
    """
    Fit TF_expr ~ C(perturbation) with NTC as reference.
    If perturbation contains '+NTC', it is simplified to remove '+NTC'.
    """

    # -------- Clean perturbation labels --------
    pert_clean = clean_perturbation_labels(adata.obs[perturb_key])

    tfs_present, tf_mat = get_tf_matrix(adata, tfs, layer=layer)

    if control not in pert_clean.unique():
        raise ValueError(f"{control} not found in perturbations")

    pert_clean = pd.Categorical(pert_clean)
    pert_clean = pert_clean.reorder_categories(
        [control] + [x for x in pert_clean.categories if x != control],
        ordered=True
    )

    results = []

    for j, tf in enumerate(tfs_present):

        y = tf_mat[:, j]

        df = pd.DataFrame({
            "y": y,
            "pert": pert_clean
        })

        model = smf.ols("y ~ C(pert)", data=df).fit(cov_type="HC3")  # robust SE

        for term in model.params.index:
            if term.startswith("C(pert)[T."):

                pert_name = term.split("T.")[1].rstrip("]")
                coef = model.params[term]
                pval = model.pvalues[term]

                results.append({
                    "perturbation": pert_name,
                    "TF": tf,
                    "effect_size": coef,
                    "pval": pval
                })

    res_df = pd.DataFrame(results)

    # FDR correction per TF
    res_df["fdr"] = np.nan
    for tf in res_df["TF"].unique():
        mask = res_df["TF"] == tf
        res_df.loc[mask, "fdr"] = bh_fdr(res_df.loc[mask, "pval"].values)

    # Keep significant only
    res_sig = res_df[res_df["fdr"] < alpha].copy()

    # Build matrix
    effect_matrix = (
        res_sig.pivot(index="perturbation",
                      columns="TF",
                      values="effect_size")
        .fillna(0)
    )

    return effect_matrix, res_df

In [ ]:
adata_dif_1=adata[adata.obs.final_label.isin(["Differentiated_1"]),:]
adata_dif_2=adata[adata.obs.final_label.isin(["Differentiated-2"]),:]

In [ ]:
res_sgn=res_sgn.loc[res_sgn.fdr < 0.0001,]

In [ ]:
res_sgn_differentiated_1 = res_sgn.loc[res_sgn.group.isin(["Differentiated_1"]),:]
res_sgn_differentiated_2 = res_sgn.loc[res_sgn.group.isin(["Differentiated-2"]),:]


In [ ]:
tfs_assoc = list(res_sgn_differentiated_1["TF"].unique())

effect_matrix_dif1, full_results_dif1 = linear_model_tf_vs_ntc(
    adata_dif_1,
    tfs=tfs_assoc,
    perturb_key="perturbation",
    control="NTC",
    layer=None,      # or "log1p"
    alpha=0.01
)


In [ ]:
tfs_assoc_2 = list(res_sgn_differentiated_2["TF"].unique())

effect_matrix_dif2, full_results_dif2 = linear_model_tf_vs_ntc(
    adata_dif_2,
    tfs=tfs_assoc_2,
    perturb_key="perturbation",
    control="NTC",
    layer=None,      # or "log1p"
    alpha=0.01
)

In [ ]:
effect_matrix_dif1.to_csv("DE_TFs_dif1.csv")
effect_matrix_dif2.to_csv("DE_TFs_dif2.csv")

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    effect_matrix,
    cmap="bwr",
    center=0,
    linewidths=0.5
)

plt.title("Significant TF Expression Changes vs NTC")
plt.xlabel("TFs")
plt.ylabel("Perturbations")
plt.tight_layout()
plt.show()

In [ ]:
print("Control used:", res_tfpert.attrs["control_label_used"])
if res_tfpert.attrs["missing_tfs"]:
    print("Missing TFs (dropped):", res_tfpert.attrs["missing_tfs"][:20])

# Significant changes (per perturbation FDR < 0.05)
res_sig = res_tfpert[res_tfpert["fdr"] < 0.05].copy()

# Summarize which perturbations change many TFs
pert_summary = (res_sig
    .assign(abs_log2fc=lambda d: np.abs(d["log2fc_mean"]))
    .groupby("perturbation")
    .agg(
        n_sig=("TF", "size"),
        n_unique_tf=("TF", "nunique"),
        median_abs_log2fc=("abs_log2fc", "median"),
    )
    .sort_values(["n_sig", "median_abs_log2fc"], ascending=False)
)

print(pert_summary.head(20))

# Save results
res_tfpert.to_csv("tf_changes_by_perturbation.tsv", sep="\t", index=False)
pert_summary.to_csv("tf_changes_by_perturbation.summary.tsv", sep="\t")